# Ad Spend Analysis: SQL
Client question: which channels drive revenue, where has spend hit diminishing returns, and how should next year's budget move?

In [43]:
import duckdb
con = duckdb.connect("../data/ads.duckdb")

for t in ["campaigns", "daily_performance", "weekly_sales", "promotions"]:
    con.sql(f"CREATE OR REPLACE TABLE {t} AS SELECT * FROM read_csv_auto('../data/{t}.csv')")

con.sql("SHOW TABLES")

┌───────────────────┐
│       name        │
│      varchar      │
├───────────────────┤
│ campaigns         │
│ daily_clean       │
│ daily_performance │
│ promotions        │
│ weekly_sales      │
└───────────────────┘

In [44]:
con.sql("DESCRIBE daily_performance").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,campaign_id,BIGINT,YES,None,None,None
2,spend,DOUBLE,YES,None,None,None
3,impressions,BIGINT,YES,None,None,None
4,clicks,BIGINT,YES,None,None,None
5,conversions,BIGINT,YES,None,None,None
6,revenue,DOUBLE,YES,None,None,None


In [45]:
con.sql("DESCRIBE weekly_sales").df()

,column_name,column_type,null,key,default,extra
0,week_start,DATE,YES,None,None,None
1,total_sales,DOUBLE,YES,None,None,None


In [46]:
con.sql("DESCRIBE promotions").df()

,column_name,column_type,null,key,default,extra
0,week_start,DATE,YES,None,None,None
1,promo_name,VARCHAR,YES,None,None,None
2,discount_pct,BIGINT,YES,None,None,None


In [47]:
con.sql("DESCRIBE campaigns").df()

,column_name,column_type,null,key,default,extra
0,campaign_id,BIGINT,YES,None,None,None
1,campaign_name,VARCHAR,YES,None,None,None
2,channel,VARCHAR,YES,None,None,None
3,region,VARCHAR,YES,None,None,None
4,language,VARCHAR,YES,None,None,None
5,creative_type,VARCHAR,YES,None,None,None
6,objective,VARCHAR,YES,None,None,None
7,start_date,DATE,YES,None,None,None
8,end_date,DATE,YES,None,None,None


# A. Data check

In [48]:
con.sql("""
SELECT 'campaigns' AS tbl, COUNT(*) AS n_rows, MIN(start_date) AS first_date, MAX(end_date) AS last_date FROM campaigns
UNION ALL
SELECT 'daily_performance', COUNT(*), MIN(date), MAX(date) FROM daily_performance
UNION ALL
SELECT 'weekly_sales', COUNT(*), MIN(week_start), MAX(week_start) FROM weekly_sales
UNION ALL
SELECT 'promotions', COUNT(*), MIN(week_start), MAX(week_start) FROM promotions
""").df()

,tbl,n_rows,first_date,last_date
0,campaigns,229,2024-01-01,2025-12-28
1,daily_performance,13062,2024-01-01,2025-12-28
2,weekly_sales,104,2024-01-01,2025-12-22
3,promotions,12,2024-02-12,2025-11-24


In [49]:
con.sql("""
-- Question: are there duplicate rows in daily_performance?
-- Grain: one row per campaign per day
-- Check: group by date, campaign_id; expect 0 rows (found 14, campaign 1137)
SELECT date, campaign_id, COUNT(*) AS n
FROM daily_performance
GROUP BY date, campaign_id
HAVING COUNT(*) > 1
""").df()

,date,campaign_id,n
0,2025-03-08,1137,2
1,2025-03-14,1137,2
2,2025-03-04,1137,2
3,2025-03-15,1137,2
4,2025-03-05,1137,2
5,2025-03-06,1137,2
6,2025-03-09,1137,2
7,2025-03-07,1137,2
8,2025-03-10,1137,2
9,2025-03-13,1137,2


In [50]:
con.sql("""
-- Question: are there duplicate rows in campaigns?
-- Grain: one row per campaign
-- Check: group by campaign_id; expect 0 rows (found 0)
SELECT campaign_id, COUNT(*) as n
FROM campaigns
GROUP BY campaign_id
HAVING COUNT(*) > 1  
""")

┌─────────────┬───────┐
│ campaign_id │   n   │
│    int64    │ int64 │
└─────────────┴───────┘
        0 rows       

In [51]:
con.sql("""
-- Question: are there duplicate rows in weekly_sales?
-- Grain: one row per week_start
-- Check: group by week_start; expect 0 rows (found 0)
SELECT week_start, COUNT(*) as n
FROM weekly_sales
GROUP BY week_start
HAVING COUNT(*) > 1  
""")

┌────────────┬───────┐
│ week_start │   n   │
│    date    │ int64 │
└────────────┴───────┘
        0 rows      

In [52]:
con.sql("""
-- Question: are there duplicate rows in promotions?
-- Grain: one row per week_start
-- Check: group by week_start; expect 0 rows (found 0)
SELECT week_start, COUNT(*) as n
FROM promotions
GROUP BY week_start
HAVING COUNT(*) > 1  
""")

┌────────────┬───────┐
│ week_start │   n   │
│    date    │ int64 │
└────────────┴───────┘
        0 rows      

In [53]:
con.sql("""
SELECT *
FROM daily_performance
WHERE campaign_id = 1137
  AND date BETWEEN '2025-03-03' AND '2025-03-16'
ORDER BY date
""").df()

,date,campaign_id,spend,impressions,clicks,conversions,revenue
0,2025-03-03,1137,606.44,10632,542,21,1655.04
1,2025-03-03,1137,606.44,10632,542,21,1655.04
2,2025-03-04,1137,650.22,12392,562,18,1369.80
3,2025-03-04,1137,650.22,12392,562,18,1369.80
4,2025-03-05,1137,508.97,8838,464,13,1078.01
5,2025-03-05,1137,508.97,8838,464,13,1078.01
6,2025-03-06,1137,579.30,9745,419,18,1553.05
7,2025-03-06,1137,579.30,9745,419,18,1553.05
8,2025-03-07,1137,593.00,10725,481,23,2010.06
9,2025-03-07,1137,593.00,10725,481,23,2010.06


In [54]:
con.sql("""
CREATE OR REPLACE VIEW daily_clean AS
SELECT DISTINCT * FROM daily_performance
""")

con.sql("SELECT COUNT(*) AS n_rows FROM daily_clean").df()

,n_rows
0,13048


- No duplicates in campaigns (by campaign_id), weekly_sales (by week_start), or promotions (by week_start).
- daily_performance: campaign 1137 has 14 days (2025-03-03 to 2025-03-16) duplicated as exact copies. Summing the raw table would double-count them. Fix: view `daily_clean` using SELECT DISTINCT. Raw table left untouched. Use daily_clean for all analysis.

In [55]:
con.sql("""
-- Question: does every daily_clean row belong to a campaign that exists?
-- Grain: one row per campaign per day
-- Check: LEFT JOIN to campaigns; expect 0 rows where c.campaign_id is NULL
SELECT dc.campaign_id, COUNT(*) AS n_days
FROM daily_clean AS dc
LEFT JOIN campaigns AS c
  ON dc.campaign_id = c.campaign_id
WHERE c.campaign_id IS NULL
GROUP BY dc.campaign_id
""").df()

,campaign_id,n_days


In [56]:
con.sql("""
-- Question: are there rows with values that can't be right?
-- Grain: one row per campaign per day
-- Check: negative spend/revenue, clicks > impressions, conversions > clicks; expect 0 of each
SELECT
  COUNT(*) FILTER (WHERE spend < 0)               AS neg_spend,
  COUNT(*) FILTER (WHERE revenue < 0)             AS neg_revenue,
  COUNT(*) FILTER (WHERE clicks > impressions)    AS clicks_gt_impr,
  COUNT(*) FILTER (WHERE conversions > clicks)    AS conv_gt_clicks
FROM daily_clean
""").df()

,neg_spend,neg_revenue,clicks_gt_impr,conv_gt_clicks
0,0,0,0,0


In [57]:
con.sql("""
-- Question: do the weekly tables start their weeks on the same day?
-- Grain: one row per week
-- Check: weekday of week_start in weekly_sales and promotions; expect all Monday
SELECT 'weekly_sales' AS tbl, dayname(week_start) AS weekday, COUNT(*) AS n
FROM weekly_sales
GROUP BY 1, 2
UNION ALL
SELECT 'promotions', dayname(week_start), COUNT(*)
FROM promotions
GROUP BY 1, 2
""").df()

,tbl,weekday,n
0,weekly_sales,Monday,104
1,promotions,Monday,12


## Findings
- campaigns, daily_performance and weekly_sales cover 2024-01-01 to 2025-12-28. weekly_sales has 104 rows = 104 weeks, no gaps (last week starts 2025-12-22).
- promotions has only 12 rows: it lists promo weeks only. Join it to weekly_sales with a LEFT JOIN, otherwise non-promo weeks are dropped.
- No duplicates in campaigns (by campaign_id) or weekly_sales (by week_start).
- daily_performance: campaign 1137 has 14 days (2025-03-03 to 2025-03-16) duplicated as exact copies. Summing the raw table would double-count them. Fix: view daily_clean using SELECT DISTINCT (13062 -> 13048 rows). Raw table left untouched; use daily_clean for all analysis.
- Orphans: 0. Every campaign_id in daily_clean exists in campaigns, so joining to get channel loses no rows.
- Impossible values: none in daily_clean (0 negative spend or revenue, 0 rows with clicks > impressions, 0 with conversions > clicks).
- Week alignment: both weekly tables start on Monday, so weekly joins line up.

# B. ROAS and CPA by campaign

In [62]:
con.sql("""
-- Question: which campaigns return the most revenue per dollar, and which cost least per conversion?
-- Grain: one row per campaign (daily rows summed first)
-- Check: ROAS and CPA are ratios of sums, not averages of daily ratios
SELECT campaign_id,
    SUM(spend) AS spend,
    SUM(conversions) AS conversions,
    SUM(revenue) AS revenue,
    SUM(revenue)/NULLIF(SUM(spend),0) AS ROAS,
    SUM(spend)/NULLIF(SUM(conversions),0) AS CPA
FROM daily_clean
GROUP BY campaign_id
ORDER BY ROAS DESC
LIMIT 20
""")

┌─────────────┬────────────────────┬─────────────┬────────────────────┬────────────────────┬────────────────────┐
│ campaign_id │       spend        │ conversions │      revenue       │        ROAS        │        CPA         │
│    int64    │       double       │   int128    │       double       │       double       │       double       │
├─────────────┼────────────────────┼─────────────┼────────────────────┼────────────────────┼────────────────────┤
│        1226 │             3409.7 │         152 │           14783.02 │  4.335577910080066 │  22.43223684210526 │
│        1187 │           20165.92 │         914 │            77927.3 │ 3.8643067115212206 │ 22.063369803063456 │
│        1032 │ 13211.509999999998 │         600 │  50856.70999999999 │ 3.8494244791095036 │  22.01918333333333 │
│        1073 │           23253.86 │        1046 │           89256.18 │  3.838338237178687 │ 22.231223709369026 │
│        1228 │            1443.92 │          59 │            5446.15 │ 3.77178098509612

## Takeaway
**Question:** Which is the better campaign, 1226 (ROAS 4.34, $3.4k spend) or 1114 (ROAS 2.97, $62k spend)?

**Answer:** It depends on the objective. For efficiency, 1226 is better (ROAS 4.34 vs 2.97). For total revenue, 1114 is better ($184k vs $15k). But these are average returns on past spend, not marginal returns: 1226 only spent $3.4k, so its ROAS may not hold if scaled, and 1114 may already be saturated. Before moving budget I would check each campaign's objective and flight length, then estimate marginal ROAS with an MMM or a spend test.